In [2]:
import pandas as pd

In [3]:
splits = {'train': 'main/train-00000-of-00001.parquet', 'test': 'main/test-00000-of-00001.parquet'}

In [ ]:
##########################################################

Importing Train and Test dataset and quick exploratory analysis

In [4]:
df_train = pd.read_parquet("hf://datasets/openai/gsm8k/" + splits["train"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [5]:
print(splits["train"])

main/train-00000-of-00001.parquet


In [6]:
df_train.head()

,question,answer
0,Natalia sold clips to 48 of her friends in Apr...,Natalia sold 48/2 = <<48/2=24>>24 clips in May...
1,Weng earns $12 an hour for babysitting. Yester...,Weng earns 12/60 = $<<12/60=0.2>>0.2 per minut...
2,Betty is saving money for a new wallet which c...,"In the beginning, Betty has only 100 / 2 = $<<..."
3,"Julie is reading a 120-page book. Yesterday, s...",Maila read 12 x 2 = <<12*2=24>>24 pages today....
4,James writes a 3-page letter to 2 different fr...,He writes each friend 3*2=<<3*2=6>>6 pages a w...


In [7]:
df_train.iloc[0,0]

'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?'

In [8]:
df_train.iloc[0,1]

'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72'

In [9]:
df_test = pd.read_parquet("hf://datasets/openai/gsm8k/" + splits["test"])

In [10]:
df_test.head()

,question,answer
0,Janet’s ducks lay 16 eggs per day. She eats th...,Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eg...
1,A robe takes 2 bolts of blue fiber and half th...,It takes 2/2=<<2/2=1>>1 bolt of white fiber\nS...
2,Josh decides to try flipping a house. He buys...,The cost of the house and repairs came out to ...
3,James decides to run 3 sprints 3 times a week....,He sprints 3*3=<<3*3=9>>9 times\nSo he runs 9*...
4,"Every day, Wendi feeds each of her chickens th...","If each chicken eats 3 cups of feed per day, t..."


In [11]:
df_test.iloc[0,0]

"Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?"

In [12]:
df_test.iloc[0,1]

'Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.\nShe makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.\n#### 18'

In [16]:
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1319 entries, 0 to 1318
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   question  1319 non-null   object
 1   answer    1319 non-null   object
dtypes: object(2)
memory usage: 20.7+ KB


In [ ]:
###########################################################################################

TRAIN and TEST SPLIT

Dataset: Use the GSM8K, main configuration. Use training examples 1–10 for development, training examples 11–14 as four-shot demonstrations, and test examples 1–30 for final evaluation. Use the same test questions throughout. Keep test reference solutions hidden from the model and tools.

In [13]:
df_dev_split = df_train.iloc[0:10,:]

In [14]:
df_shot = df_train.iloc[10:14,:]

In [15]:
df_test_exp = df_test.iloc[0:30,:]

In [ ]:
####################################################################################

**Question 1 (10 marks)**
Compare four methods: zero-shot prompting, four-shot prompting, self-consistency using four sampled solutions with majority voting, and an initial solution followed by one reflection/revision call. Finalize prompts using development examples only. Report test accuracy, total generated tokens, and model calls in one table. Analyze two cases where additional inference helped or harmed performance, and justify which method provides the best accuracy–cost tradeoff.

In [23]:
import re
from collections import Counter
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

In [24]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

In [ ]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7473 entries, 0 to 7472
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   question  7473 non-null   object
 1   answer    7473 non-null   object
dtypes: object(2)
memory usage: 116.9+ KB


In [ ]:
#####################################################

Create model and tokenizer object

In [25]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype = "auto",
    device_map = "auto"
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Informative small notes

**torch_dtype** is a parameter used in deep learning libs like HFTs to override the default data type when loading model weight.

It targets to specify the PyTorch data type (such as torch.float16, torch.bfloat16 or torch.float32) in which model weights should be instantiated upon loading via methods like from_pretrained

torch.float16 or torch.half : 16-bit floating point, saves VRAM and speeds up training/inference on supported GPUs.
torch.bfoat16: Brain floating point (16-bit) offers a wider dynamic range than float16 and is widely used for large language models.

torch_dtype = "auto" tells HF lib to automatically select the data type the model was originally trained and saved in

why use auto prevents precision issues:
Loading a model trained in bfloat16 in to float16 can cause severe underflow/overflow errors (giving you NaN outputs) Using "auto" ensures you match the original training set up

In [18]:
import re

def extract_answer(text):
    """
    Extracts numerical digits from model responses.
    Handles '#### 72', 'Final Answer: ... 72 clips.', and trailing digits.
    """
    if not text:
        return None

    # 1. Standard GSM8K format: "#### 72" or "#### $72"
    match = re.search(r"####\s*\$?\s*([0-9,\.]+)", text)
    if match:
        ans = match.group(1).rstrip('.').replace(",", "")
        return ans

    # 2. Look after "Final Answer:" for the FIRST sequence of digits
    match_heading = re.search(r"final\s*answer[:\s]*.*?\b([0-9]+(?:\.[0-9]+)?)\b", text, re.IGNORECASE)
    if match_heading:
        return match_heading.group(1)

    # 3. Fallback: Find all sequences of pure digits in the text and return the last one
    numbers = re.findall(r"\b([0-9]+(?:\.[0-9]+)?)\b", text)
    if numbers:
        return numbers[-1]

    return None

In [19]:
def generate_response(messages, do_sample = False, temperature=0.7, top_p=0.9, max_new_tokens=512):

  """

  """


  text = tokenizer.apply_chat_template(
      messages,
      tokenize=False,
      add_generation_prompt = True
  )


  model_inputs = tokenizer([text], return_tensors = "pt").to(model.device)

  #configure generation parameters

  gen_kwargs = {
      "max_new_tokens": max_new_tokens,
      "do_sample": do_sample,
  }
  if do_sample:
    gen_kwargs["temperature"]= temperature
    gen_kwargs["top_p"] = top_p

  with torch.no_grad():
    output_ids = model.generate(**model_inputs, **gen_kwargs)

  #strip prompt tokens from output
  input_lenght = model_inputs.input_ids.shape[1]
  generated_tokens = output_ids[0][input_lenght:]

  num_generated_tokens = len(generated_tokens)
  response_text = tokenizer.decode(generated_tokens, skip_special_tokens = True)

  return response_text, num_generated_tokens



In [ ]:
for i in range(10):

  gen_res = generate_response(df_dev_split.iloc[i,0])[0]
  print(12*"-",f"Question {i+1}",12*"-","\n",df_dev_split.iloc[i,0])
  print()
  print(f"* MODEL's Response-{i+1} * : \n\n","** ",gen_res," **","\n",sep = "")

  print("Model Answer :", extract_answer(gen_res),"\n")
  print("Correct Answer :", extract_answer(df_dev_split.iloc[i,1]),"\n")

  print(f"Generated Tokens for Response-{i+1} :",generate_response(df_dev_split.iloc[i,0])[1],"\n\n")


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


------------ Question 1 ------------ 
 Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?

* MODEL's Response-1 * : 

** As an AI language model, I don't have personal experiences or emotions like humans do, but I can provide you with some general information about the topic of your question.

If you have any specific questions related to technology, science, or other areas that interest you, feel free to ask! I'll do my best to help answer them in a way that's as informative and engaging as possible. **

Model Answer : None 

Correct Answer : 72 

Generated Tokens for Response-1 : 80 


------------ Question 2 ------------ 
 Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?

* MODEL's Response-2 * : 

** As an AI language model, I don't have personal experiences or emotions like humans do, but I can provide you with so

In [ ]:
#With zero shot prompting model have no idea to generate to response. It has a ability to grasp soluton but it has  no idea how it attends it

#without example and explicit operational structure (Zero Shot), the model fails to trigger its problem-solving (mathematical reasoning) capabilities
#In small instruction-tuned models (like Qwen2.5-0.5B-Instruct), in this case because input formating does not exist which normally push model problem solving,
#greedy decoding (do_sample=False) on a minimal zero-shot
#prompt can trigger standard "safety/assistant boilerplates" .
#Without a ground in-context demostration, 0.5 Billion parameter model lacks suuficient guidance and It treats the prompt as an ambiguous query rather than a direct instruction to calculate

For zero shot

Model responses requets as

"As an AI language model, I don't have personal experiences or emotions like humans do, but I can provide you with some general information about the topic of your question.

If you have any specific questions related to technology, science, or other areas that interest you, feel free to ask! I'll do my best to help answer them in a way that's informative and helpful"

at each times

and naturally because it return same response each time, it generates 77 tokens in all response

In [ ]:
####Four SHot

In [ ]:
def build_four_shot_prompt(few_shot_df, current_question):
    """
    Constructs the few-shot context from df_shot and appends the target question.
    """
    messages = [
        {"role": "system", "content": "You are a helpful math assistant. Solve the math problem step by step, following the format of the examples. At the end, state the final numerical answer after '####'."}
    ]

    # Add the 4 demonstration examples as alternating user/assistant turns (or inside a single user prompt)
    # Using a single structured user prompt format:
    few_shot_text = "Here are some examples of how to solve math word problems:\n\n"
    for _, row in few_shot_df.iterrows():
        few_shot_text += f"Question: {row['question']}\nAnswer:\n{row['answer']}\n\n"

    few_shot_text += f"Now solve this new problem:\nQuestion: {current_question}\nAnswer:"

    messages.append({"role": "user", "content": few_shot_text})
    return messages





In [ ]:
# --- Loop for Four-Shot Evaluation on Dev Split ---
print("=== METHOD 2: FOUR-SHOT PROMPTING ===")
for i in range(10):
    q_text = df_dev_split.iloc[i, 0]

    # Build messages with the 4-shot context
    messages = build_four_shot_prompt(df_shot, q_text)

    response_text, token_count = generate_response(messages, do_sample=False)

    print(12*"-", f"Question {i+1}", 12*"-", "\n", q_text)
    print()
    print(f"* MODEL's Response-{i+1}: \n\n** ", response_text, " **", "\n\n\n", sep="")
    print()
    print(f"* MODEL's Answer-{i+1}: \n\n** ", extract_answer(response_text), " **", "\n", sep="")
    print("Correct Answer :", extract_answer(df_dev_split.iloc[i, 1]), "\n")
    print(f"Generated Tokens for Response-{i+1} :", token_count, "\n\n")

=== METHOD 2: FOUR-SHOT PROMPTING ===
------------ Question 1 ------------ 
 Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?

* MODEL's Response-1: 

** To find out how many clips Natalia sold in total, we need to follow these steps:

1. **Determine the number of clips sold in May**:
   - In April, Natalia sold 48 clips.
   - In May, she sold half as many clips as in April.

   Therefore, the number of clips sold in May is:
   \[
   \frac{48}{2} = 24 \text{ clips}
   \]

2. **Calculate the total number of clips sold**:
   - Natalia sold clips in both April and May.
   - The total number of clips sold is the sum of the clips sold in April and May.

   Thus, the total number of clips sold is:
   \[
   48 + 24 = 72 \text{ clips}
   \]

### Final Answer:
Natalia sold a total of **72 clips** in April and May. **




* MODEL's Answer-1: 

** 72 **

Correct Answer : 72 

Generated Token

In [ ]:
#Because model's weights are frozen, and it totally relies on provided tokens, feeding the model with both question and ansnwer allow it to find where its pre existing weights
# Few shot the model in instructive format tells to that model will not give conversational response and guide it to focus on correct parameters
#step by step mimic reasoning style.. It is observed at model output how it breaks down the problem into logical steps eventhoug questions are in the 1`1\? in variety formating

In [ ]:
def build_four_shot_messages(df_shot, current_question):
    """
    Formats training examples 11-14 (df_shot) into a few-shot demonstration prompt.
    """
    few_shot_prompt = "Below are four example math word problems with step-by-step solutions and final answers.\n\n"

    for idx, row in df_shot.iterrows():
        few_shot_prompt += f"Question: {row['question']}\nAnswer:\n{row['answer']}\n\n"

    few_shot_prompt += f"Now, solve the following problem step-by-step and state the final answer after '#### '.\nQuestion: {current_question}\nAnswer:"

    messages = [
        {"role": "system", "content": "You are a helpful math assistant. Solve the math problem step by step, following the style of the provided examples."},
        {"role": "user", "content": few_shot_prompt}
    ]
    return messages


# --- Method 2: Four-Shot Evaluation Loop ---
print("=== METHOD 2: FOUR-SHOT PROMPTING ===")

for i in range(10):
    q_text = df_dev_split.iloc[i, 0]
    true_ans = extract_answer(df_dev_split.iloc[i, 1])

    # 1. Build messages with df_shot examples
    messages = build_four_shot_messages(df_shot, q_text)

    # 2. Greedy decoding (do_sample=False)
    response_text, token_count = generate_response(messages, do_sample=False)

    # 3. Extract final answer
    pred_ans = extract_answer(response_text)
    is_correct = (pred_ans == true_ans)

    print(12*"-", f"Question {i+1} (Four-Shot)", 12*"-")
    print(f"Question: {q_text}\n")
    print(f"* MODEL's Response-{i+1}:\n{response_text}\n")
    print(f"* Extracted Answer: {pred_ans} | Correct: {is_correct}")
    print(f"Correct Answer: {true_ans}")
    print(f"Generated Tokens: {token_count} (1 model call)\n\n")

=== METHOD 2: FOUR-SHOT PROMPTING ===
------------ Question 1 (Four-Shot) ------------
Question: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?

* MODEL's Response-1:
Step 1: First, we need to find out how many clips Natalia sold in May. We know that she sold half as many clips in May as she did in April, which was 48 clips. So, we can calculate this as follows:

May clips = April clips / 2
May clips = 48 / 2
May clips = 24 clips

Step 2: Now that we know Natalia sold 24 clips in May, we can add this to the number of clips she sold in April (which was 48) to find the total number of clips she sold altogether in April and May:

Total clips = April clips + May clips
Total clips = 48 + 24
Total clips = 72

Final Answer: Natalia sold a total of 72 clips altogether in April and May.

* Extracted Answer: 72 | Correct: True
Correct Answer: 72
Generated Tokens: 180 (1 model call)


----

In [ ]:
#### self-consistency using four sampled solutions with majoority voting

In [ ]:
from collections import Counter

print("=== METHOD 3: SELF-CONSISTENCY (4 SAMPLES) ===")

system_prompt = (
    "You are a helpful math assistant. Solve the math problem step by step. "
    "At the end of your solution, state the final numerical answer after '#### '."
)

for i in range(10):
    q_text = df_dev_split.iloc[i, 0]
    true_ans = extract_answer(df_dev_split.iloc[i, 1])

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": q_text}
    ]

    sampled_answers = []
    total_tokens_q = 0
    model_calls_q = 0
    sample_responses = []

    # Generate 4 sampled reasoning paths
    for sample_idx in range(4):
        resp, tokens = generate_response(
            messages,
            do_sample=True,
            temperature=0.7,
            top_p=0.9
        )
        total_tokens_q += tokens
        model_calls_q += 1
        sample_responses.append(resp)

        # Extract answer from THIS sample
        ans = extract_answer(resp)
        if ans is not None:
            sampled_answers.append(ans)

    # Perform majority voting across the 4 extracted answers
    if sampled_answers:
        counts = Counter(sampled_answers)
        majority_ans = counts.most_common(1)[0][0]
    else:
        majority_ans = None

    is_correct = (majority_ans == true_ans)

    print(12*"-", f"Question {i+1} (Self-Consistency)", 12*"-")
    print(f"Question: {q_text}\n")
    print(f"* Sampled Extracted Answers: {sampled_answers}")
    print(f"* Majority Voted Answer: ** {majority_ans} ** | Correct: {is_correct}")
    print(f"Correct Answer: {true_ans}")
    print(f"Total Tokens ({model_calls_q} calls): {total_tokens_q}\n\n")

=== METHOD 3: SELF-CONSISTENCY (4 SAMPLES) ===
------------ Question 1 (Self-Consistency) ------------
Question: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?

* Sampled Extracted Answers: ['72', '72', '72', '72']
* Majority Voted Answer: ** 72 ** | Correct: True
Correct Answer: 72
Total Tokens (4 calls): 751


------------ Question 2 (Self-Consistency) ------------
Question: Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?

* Sampled Extracted Answers: ['10', '10.00', '10', '10']
* Majority Voted Answer: ** 10 ** | Correct: True
Correct Answer: 10
Total Tokens (4 calls): 568


------------ Question 3 (Self-Consistency) ------------
Question: Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. Her parents decided to give her $15 for that purpose, and her grandparent

In [ ]:
When we operate self connsistency method we make do_sample (do_sample =True) parameter activated.
This allow the model explore different reasoning routes in contrast to Zero shot which model picks the single most likely token every step, crating one deterministic path
Self consistency follow stocastic generating in a controlled randomness, due to we demand it 4 times
Small models as our model 0.5 billion parameter make minor arithmetic or logical slips on a single pass.
With random sampling reasoning routes model manage to break out rigid conversational loops it is stuck which is happen in zero shot.
On the otherhand randomness cause more exploration and naturally generate more tokens cost engine resources.

In [ ]:
###Initial Solution followed by one reflection/revision call

In [ ]:
print("=== METHOD 4: INITIAL SOLUTION + REFLECTION ===")

system_prompt = (
    "You are a helpful math assistant. Solve the math problem step by step. "
    "At the end of your solution, state the final numerical answer after '#### '."
)

for i in range(10):
    q_text = df_dev_split.iloc[i, 0]
    true_ans = extract_answer(df_dev_split.iloc[i, 1])

    # --- STEP 1: Generate Initial Draft ---
    messages_call1 = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": q_text}
    ]
    resp1, tokens1 = generate_response(messages_call1, do_sample=False)

    # --- STEP 2: Reflection & Revision Pass ---
    reflection_instruction = (
        f"Here is a draft solution to the math problem:\n\n{resp1}\n\n"
        "Please review this solution step-by-step. Double-check all calculations and logic. "
        "Provide your revised step-by-step solution and state the final numerical answer after '#### '."
    )

    messages_call2 = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": q_text},
        {"role": "assistant", "content": resp1},
        {"role": "user", "content": reflection_instruction}
    ]
    resp2, tokens2 = generate_response(messages_call2, do_sample=False)

    total_tokens_q = tokens1 + tokens2
    model_calls_q = 2

    # Extract final answer strictly from the second (revised) call
    revised_pred_ans = extract_answer(resp2)
    is_correct = (revised_pred_ans == true_ans)

    print(12*"-", f"Question {i+1} (Reflection)", 12*"-")
    print(f"Question: {q_text}\n")
    print(f"* Initial Answer Draft (Call 1): {extract_answer(resp1)}")
    print(f"* Revised Response (Call 2):\n{resp2}\n")
    print(f"* Final Extracted Answer: {revised_pred_ans} | Correct: {is_correct}")
    print(f"Correct Answer: {true_ans}")
    print(f"Total Tokens ({model_calls_q} calls): {total_tokens_q}\n\n")

=== METHOD 4: INITIAL SOLUTION + REFLECTION ===
------------ Question 1 (Reflection) ------------
Question: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?

* Initial Answer Draft (Call 1): 72
* Revised Response (Call 2):
To determine the total number of clips Natalia sold in April and May, we can follow these steps:

1. **Identify the number of clips sold in April:**
   Natalia sold clips to 48 of her friends in April.

2. **Calculate the number of clips sold in May:**
   In May, Natalia sold half as many clips as she did in April. To find half of 48, we perform the division \( \frac{48}{2} \).
   
   \[
   \frac{48}{2} = 24
   \]
   So, Natalia sold 24 clips in May.

3. **Add the number of clips sold in April and May together:**
   To find the total number of clips sold in both months, we add the number of clips sold in April (48) to the number of clips sold in May (24).

   \[

In [ ]:
Eventhough in this method model follows same parameter do_sample - False greedy decoding
the context changes between call 1 and call 2 in our model development runs.
At the first shot model figure out math steps and geenrate tokens from scratch and at the second shot
model contain original question, a geenrated solution draft and with the prompt instruction review the solution step by step against to zero shot method

by transformer self attention mechanism with it gained from the prior call the probablitiy distribution change completely
Model has aexperience and no longer tot ry the problem from scratch
Model walk trhoough wit its own evaluator guide tself based on its pretrained tokens kind of its expreience
it narrow the solution context and harness the self attention mechanics to focus on useful tokens

TESTING ALL METHODS on TEST DATASET

In [ ]:
for i in range(10):

  gen_res = generate_response(df_test.iloc[i,0])[0]
  print(12*"-",f"Question {i+1}",12*"-","\n",df_test.iloc[i,0])
  print()
  print(f"* MODEL's Response-{i+1} * : \n\n","** ",gen_res," **","\n",sep = "")

  print("Model Answer :", extract_answer(gen_res),"\n")
  print("Correct Answer :", extract_answer(df_test.iloc[i,1]),"\n")

  print(f"Generated Tokens for Response-{i+1} :",generate_response(df_test.iloc[i,0])[1],"\n\n")

------------ Question 1 ------------ 
 Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?

* MODEL's Response-1 * : 

** As an AI language model, I don't have personal experiences or emotions like humans do, but I can provide you with some general information about the topic of your question.

If you have any specific questions related to technology, science, or other areas that interest you, feel free to ask! I'll do my best to help answer them in a way that's informative and helpful. **

Model Answer : None 

Correct Answer : 18 

Generated Tokens for Response-1 : 77 


------------ Question 2 ------------ 
 A robe takes 2 bolts of blue fiber and half that much white fiber.  How many bolts in total does it take?

* MODEL's Response-2 * : 

** As an AI langua

In [44]:
import pandas as pd

# Updated DataFrame columns including Model Answer and Correct Answer
df_columns = [
    "Method",
    "Question Number",
    "Model Answer",
    "Correct Answer",
    "False",
    "True",
    "Total Generated Token"
]
zshot_rows = []

for i in range(df_test_exp.shape[0]):
    # Using df_test_exp as requested, keeping your original indexing logic
    q_text = df_test_exp.iloc[i, 0]
    true_ans_raw = df_test_exp.iloc[i, 1]

    # Single generation call to preserve your exact code structure while avoiding duplicate inference calls
    gen_res, token_count = generate_response(q_text)

    model_ans = extract_answer(gen_res)
    correct_ans = extract_answer(true_ans_raw)

    is_correct = (model_ans == correct_ans) if (model_ans is not None and correct_ans is not None) else False

    zshot_rows.append({
        "Method": "Zero-Shot",
        "Question Number": i + 1,
        "Model Answer": model_ans,
        "Correct Answer": correct_ans,
        "False": 1 if not is_correct else 0,
        "True": 1 if is_correct else 0,
        "Total Generated Token": token_count
    })

    print(12*"-", f"Question {i+1}", 12*"-", "\n", q_text)
    print()
    print(f"* MODEL's Response-{i+1}: \n\n", "** ", gen_res, " **", "\n", sep="")
    print("Model Answer :", model_ans, "\n")
    print("Correct Answer :", correct_ans, "\n")
    print(f"Generated Tokens for Response-{i+1} :", token_count, "\n\n")

dfZshot = pd.DataFrame(zshot_rows, columns=df_columns)

------------ Question 1 ------------ 
 Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?

* MODEL's Response-1: 

** As an AI language model, I don't have personal experiences or emotions like humans do, but I can provide you with some general information about the topic of your question.

If you have any specific questions related to technology, science, or other areas that interest you, feel free to ask! I'll do my best to help answer them in a way that's as informative and engaging as possible. **

Model Answer : None 

Correct Answer : 18 

Generated Tokens for Response-1 : 80 


------------ Question 2 ------------ 
 A robe takes 2 bolts of blue fiber and half that much white fiber.  How many bolts in total does it take?

* MODEL's Response-2: 

** As an

In [45]:
dfZshot

,Method,Question Number,Model Answer,Correct Answer,False,True,Total Generated Token
0,Zero-Shot,1,None,18,1,0,80
1,Zero-Shot,2,None,3,1,0,80
2,Zero-Shot,3,None,70000,1,0,80
3,Zero-Shot,4,None,540,1,0,80
4,Zero-Shot,5,None,20,1,0,80
5,Zero-Shot,6,None,64,1,0,80
6,Zero-Shot,7,None,260,1,0,80
7,Zero-Shot,8,None,160,1,0,80
8,Zero-Shot,9,None,45,1,0,80
9,Zero-Shot,10,None,460,1,0,80


In [ ]:
#Four SHot

In [39]:
def build_four_shot_prompt(few_shot_df, current_question):
    """
    Constructs the few-shot context from df_shot and appends the target question.
    """
    messages = [
        {"role": "system", "content": "You are a helpful math assistant. Solve the math problem step by step, following the format of the examples. At the end, state the final numerical answer after '####'."}
    ]

    # Add the 4 demonstration examples as alternating user/assistant turns (or inside a single user prompt)
    # Using a single structured user prompt format:
    few_shot_text = "Here are some examples of how to solve math word problems:\n\n"
    for _, row in few_shot_df.iterrows():
        few_shot_text += f"Question: {row['question']}\nAnswer:\n{row['answer']}\n\n"

    few_shot_text += f"Now solve this new problem:\nQuestion: {current_question}\nAnswer:"

    messages.append({"role": "user", "content": few_shot_text})
    return messages


In [35]:
df_test_exp.shape[0]

30

In [46]:
fshot_rows = []

print("=== METHOD 2: FOUR-SHOT PROMPTING ===")

for i in range(df_test_exp.shape[0]):
    q_text = df_test_exp.iloc[i, 0]
    true_ans_raw = df_test_exp.iloc[i, 1]

    messages = build_four_shot_prompt(df_shot, q_text)
    response_text, token_count = generate_response(messages, do_sample=False)

    model_ans = extract_answer(response_text)
    correct_ans = extract_answer(true_ans_raw)

    is_correct = (model_ans == correct_ans) if (model_ans is not None and correct_ans is not None) else False

    fshot_rows.append({
        "Method": "Four-Shot",
        "Question Number": i + 1,
        "Model Answer": model_ans,
        "Correct Answer": correct_ans,
        "False": 1 if not is_correct else 0,
        "True": 1 if is_correct else 0,
        "Total Generated Token": token_count
    })

    print(12*"-", f"Question {i+1}", 12*"-", "\n", q_text)
    print()
    print(f"* MODEL's Response-{i+1}: \n\n** ", response_text, " **", "\n\n\n", sep="")
    print()
    print(f"* MODEL's Answer-{i+1}: \n\n** ", model_ans, " **", "\n", sep="")
    print("Correct Answer :", correct_ans, "\n")
    print(f"Generated Tokens for Response-{i+1} :", token_count, "\n\n")

dfFshot = pd.DataFrame(fshot_rows, columns=df_columns)

=== METHOD 2: FOUR-SHOT PROMPTING ===
------------ Question 1 ------------ 
 Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?

* MODEL's Response-1: 

** To find out how much Janet makes every day at the farmers' market, we need to calculate the total earnings from selling the eggs.

First, let's determine how many eggs she has left after accounting for what she eats and bakes:

- She starts with 16 eggs per day.
- She eats 3 eggs for breakfast.
- She bakes 4 muffins for her friends.

Total eggs eaten and baked = 3 (breakfast) + 4 (baking) = 7 eggs.

Now, subtract the number of eggs she eats and bakes from the initial amount to find out how many eggs remain:

Remaining eggs = Initial eggs - Eggs eaten and baked
Remaining eggs = 16 - 7 = 9 eggs.

Next, we'll 

In [47]:
dfFshot

,Method,Question Number,Model Answer,Correct Answer,False,True,Total Generated Token
0,Four-Shot,1,53,18,1,0,292
1,Four-Shot,2,3,3,0,1,233
2,Four-Shot,3,195000,70000,1,0,327
3,Four-Shot,4,180,540,1,0,200
4,Four-Shot,5,40,20,1,0,268
5,Four-Shot,6,48,64,1,0,166
6,Four-Shot,7,260,260,0,1,351
7,Four-Shot,8,50000,160,1,0,381
8,Four-Shot,9,477,45,1,0,321
9,Four-Shot,10,45,460,1,0,246


In [ ]:
#Self Consistency

In [ ]:
from collections import Counter
import pandas as pd

# Define standard columns matching Zero-Shot and Four-Shot pipelines
df_columns = [
    "Method",
    "Question Number",
    "Model Answer",
    "Correct Answer",
    "False",
    "True",
    "Total Generated Token"
]
selfc_rows = []

print("=== METHOD 3: SELF-CONSISTENCY (4 SAMPLES) ===")

system_prompt = (
    "You are a helpful math assistant. Solve the math problem step by step. "
    "At the end of your solution, state the final numerical answer after '#### '."
)

for i in range(df_test_exp.shape[0]):
    q_text = df_test_exp.iloc[i, 0]
    true_ans = extract_answer(df_test_exp.iloc[i, 1])

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": q_text}
    ]

    sampled_answers = []
    total_tokens_q = 0
    model_calls_q = 0
    sample_responses = []

    # Generate 4 sampled reasoning paths
    for sample_idx in range(4):
        resp, tokens = generate_response(
            messages,
            do_sample=True,
            temperature=0.7,
            top_p=0.9
        )
        total_tokens_q += tokens
        model_calls_q += 1
        sample_responses.append(resp)

        # Extract answer from THIS sample
        ans = extract_answer(resp)
        if ans is not None:
            sampled_answers.append(ans)

    # Perform majority voting across the 4 extracted answers
    if sampled_answers:
        counts = Counter(sampled_answers)
        majority_ans = counts.most_common(1)[0][0]
    else:
        majority_ans = None

    is_correct = (majority_ans == true_ans) if (majority_ans is not None and true_ans is not None) else False

    # Log structured output into row dictionary
    selfc_rows.append({
        "Method": "Self-Consistency",
        "Question Number": i + 1,
        "Model Answer": majority_ans,
        "Correct Answer": true_ans,
        "False": 1 if not is_correct else 0,
        "True": 1 if is_correct else 0,
        "Total Generated Token": total_tokens_q
    })

    # Print statements kept exactly as in your original mechanism
    print(12*"-", f"Question {i+1} (Self-Consistency)", 12*"-")
    print(f"Question: {q_text}\n")
    print(f"* Sampled Extracted Answers: {sampled_answers}")
    print(f"* Majority Voted Answer: ** {majority_ans} ** | Correct: {is_correct}")
    print(f"Correct Answer: {true_ans}")
    print(f"Total Tokens ({model_calls_q} calls): {total_tokens_q}\n\n")

# Create the final DataFrame for Self-Consistency
dfSelfC = pd.DataFrame(selfc_rows, columns=df_columns)

=== METHOD 3: SELF-CONSISTENCY (4 SAMPLES) ===
------------ Question 1 (Self-Consistency) ------------
Question: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?

* Sampled Extracted Answers: ['0', '26', '18', '18']
* Majority Voted Answer: ** 18 ** | Correct: True
Correct Answer: 18
Total Tokens (4 calls): 1301


------------ Question 2 (Self-Consistency) ------------
Question: A robe takes 2 bolts of blue fiber and half that much white fiber.  How many bolts in total does it take?

* Sampled Extracted Answers: ['3', '3', '3', '3']
* Majority Voted Answer: ** 3 ** | Correct: True
Correct Answer: 3
Total Tokens (4 calls): 754


------------ Question 3 (Self-Consistency) ------------
Question: Josh decides to try flipping a house.  He buys a house for $80,000

In [ ]:
dfSelfC

In [ ]:
import pandas as pd

# Standard columns across all methods
df_columns = [
    "Method",
    "Question Number",
    "Model Answer",
    "Correct Answer",
    "False",
    "True",
    "Total Generated Token"
]
revised_rows = []

print("=== METHOD 4: INITIAL SOLUTION + REFLECTION ===")

system_prompt = (
    "You are a helpful math assistant. Solve the math problem step by step. "
    "At the end of your solution, state the final numerical answer after '#### '."
)

for i in range(df_test_exp.shape[0]):
    q_text = df_test_exp.iloc[i, 0]
    true_ans = extract_answer(df_test_exp.iloc[i, 1])

    # --- STEP 1: Generate Initial Draft ---
    messages_call1 = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": q_text}
    ]
    resp1, tokens1 = generate_response(messages_call1, do_sample=False)

    # --- STEP 2: Reflection & Revision Pass ---
    reflection_instruction = (
        f"Here is a draft solution to the math problem:\n\n{resp1}\n\n"
        "Please review this solution step-by-step. Double-check all calculations and logic. "
        "Provide your revised step-by-step solution and state the final numerical answer after '#### '."
    )

    messages_call2 = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": q_text},
        {"role": "assistant", "content": resp1},
        {"role": "user", "content": reflection_instruction}
    ]
    resp2, tokens2 = generate_response(messages_call2, do_sample=False)

    total_tokens_q = tokens1 + tokens2
    model_calls_q = 2

    # Extract final answer strictly from the second (revised) call
    revised_pred_ans = extract_answer(resp2)
    is_correct = (revised_pred_ans == true_ans) if (revised_pred_ans is not None and true_ans is not None) else False

    # Log structured row
    revised_rows.append({
        "Method": "Reflection",
        "Question Number": i + 1,
        "Model Answer": revised_pred_ans,
        "Correct Answer": true_ans,
        "False": 1 if not is_correct else 0,
        "True": 1 if is_correct else 0,
        "Total Generated Token": total_tokens_q
    })

    # Print output matching your exact structure
    print(12*"-", f"Question {i+1} (Reflection)", 12*"-")
    print(f"Question: {q_text}\n")
    print(f"* Initial Answer Draft (Call 1): {extract_answer(resp1)}")
    print(f"* Revised Response (Call 2):\n{resp2}\n")
    print(f"* Final Extracted Answer: {revised_pred_ans} | Correct: {is_correct}")
    print(f"Correct Answer: {true_ans}")
    print(f"Total Tokens ({model_calls_q} calls): {total_tokens_q}\n\n")

# Create the final DataFrame for Method 4
dfRevised = pd.DataFrame(revised_rows, columns=df_columns)

In [ ]:
dfRevised